In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# OLD8 局部时间状态：双空间副本验证

直接 Run all。固定四臂 OFF／PAYLOAD_MULTI／STATE_OLD_MULTI／STATE_REPLICA_MULTI，共享同一来源、种子、初始状态与各自新复制的原生 scheduler，各运行50步，控制作用于25–49步。每个时间窗的第二组四个8×8块复制第一组的逻辑时间状态；不同时间位置仍携带不同状态。频率、实余弦基、RM32状态码及重复载荷算法不变。

组A保留原四块；组B将各块横向移动8个潜空间采样点，八块不重叠，位置在运行前固定。两副本目标各为原alpha/√2，总目标L2仍为1。联合11136个active系数取mean-MSE，eta1392抵消坐标数翻倍。先完成原OLD8，同次保存第25–49步实际物理pilot范数；新臂联合梯度逐步缩放至该范数，允许放大，明确区别于旧shrink-only cap。组间实际能量不必各半；保留每组能量、缩放和匹配误差。配对预算只用于写端，不提供给盲接收器。载荷规则不变但实际载荷增量可能变化，完整merged更新不宣称等能量。

盲接收按公开对应分别提取A/B逐窗口软响应，各乘√2后等权合并，继续用原alpha、ABS/DIFF评分、174条合法路径及固定等价类。合并使用支持交集，不按真值、消息、表现选组或权重，不跨时间平均。OLD8单组和REPLICA8合并两个公开接收协议均读取全部臂、通道和正确/错误密钥，预定比较为原臂的OLD8读出与新臂的REPLICA8读出，交叉结果全部保留；ABS/DIFF分别报告，不能事后择优宣称通过。组原始响应、支持、接收窗口坐标及合并响应全部保留；副本不是独立样本。

每臂真实VAE解码后保存一份RGB8，分别形成DIRECT_RGB8、RAW420、MP4，共12次重编码。主盲读取48份q、96份评分、16704条路径成本；24次重复载荷读取和48条消息后评。100份写端侧车与原生终态16q／32评分另存，终态不作为媒体检测输入。当前只运行完整视频g0/R44诊断；不把公开候选目录中含编辑模型等同于真实裁剪、删除、重复和速度变化已经验证。

首轮检验固定状态能量下的空间冗余与软合并能否保留时间证据，不预设提高信噪比。若终态失败，核对写入与归一化；若DIRECT正确而MP4失败，按逐组响应定位损失并结束当前候选判断，不自动加强度、加副本、换频率或改同步算法。只有MP4正确时间对应或预定等价类可被盲接收区分，才推进后续编辑同步与时间相关载荷；排名略有改善不算完成。

EXECUTION_COMPLETE只表示执行完整，所有acceptance仍未校准。重复载荷零误码不证明同步帮助恢复，画质仅保留18组成对诊断。沿用当前Python和已跑通的依赖探测，不强制GPU型号或新建venv。真实GPU／VAE／媒体实验由用户运行。Published source: 0f37b519334acdcaca07d35ce120385c983f2aa6.


In [ ]:
SOURCE_SHA = '0f37b519334acdcaca07d35ce120385c983f2aa6'
FIXED = {'source_cases': 1, 'arms': 4, 'channels': 3, 'reader_families': 2, 'generated': 4, 'generation_steps': 200, 'controlled_steps': 75, 'writer_sidecars': 100, 'underlying_observations': 12, 'normalized': 12, 'keys': 2, 'keyed_projections': 48, 'path_score_records': 96, 'valid_paths': 174, 'catalog_per_record': 3915, 'path_costs': 16704, 'difference_edge_costs': 272448, 'absolute_local_state_costs': 95040, 'path_posthoc': 96, 'difference_edge_posthoc': 48, 'payload_reads': 24, 'payload_evaluations': 48, 'catalog_slots': 375840, 'structural_exclusions': 359136, 'quality_pairs': 18, 'group_observation_records': 24}
TERMINAL_FIXED = {'inputs': 4, 'keyed_projections': 16, 'path_score_records': 32, 'path_costs': 5568, 'difference_edge_costs': 90816, 'absolute_local_state_costs': 31680, 'path_posthoc': 32, 'difference_edge_posthoc': 16, 'catalog_slots': 125280, 'structural_exclusions': 119712, 'group_observation_records': 8}
from pathlib import Path
import datetime,hashlib,json,os,subprocess,sys,traceback
STAMP=datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUTPUT=Path('/content/drive/MyDrive/Video-WM/Video-Local-Fourier-RM-Replica-V1')/STAMP
OUTPUT.mkdir(parents=True,exist_ok=False)
RUN_OUTPUT=OUTPUT/'fixed_reference'
REPO=Path('/content/SC-SSTW-LOCAL-FOURIER-RM-REPLICA-'+STAMP)
PYTHON=sys.executable
ARMS=('OFF','PAYLOAD_MULTI','STATE_OLD_MULTI','STATE_REPLICA_MULTI');KEYS=('CORRECT','WRONG')
CHANNELS=('DIRECT_RGB8','RAW420','MP4');FAMILIES=('OLD8','REPLICA8');MODES=('ABSOLUTE_CONTROL','ADJACENT_DIFFERENCE')
setup=dict(status='SETUP_STARTED',source_sha=SOURCE_SHA,fixed_denominator=FIXED,
 generation={a:{'status':'NOT_RUN_SETUP'} for a in ARMS},writer_diagnostics={a+'/'+str(i):{'status':'NOT_RUN_SETUP'} for a in ARMS for i in range(25,50)},
 terminal_inputs={a:{'status':'NOT_RUN_SETUP'} for a in ARMS},rasters={a:{'status':'NOT_RUN_SETUP'} for a in ARMS},
 transport={a+'/'+c:{'status':'NOT_RUN_SETUP'} for a in ARMS for c in CHANNELS},normalized={a+'/'+c:{'status':'NOT_RUN_SETUP'} for a in ARMS for c in CHANNELS},
 projections={a+'/'+c+'/'+f+'/'+k:{'status':'NOT_RUN_SETUP'} for a in ARMS for c in CHANNELS for f in FAMILIES for k in KEYS},
 mode_reads={a+'/'+c+'/'+f+'/'+k+'/'+m:{'status':'NOT_RUN_SETUP','accepted_payload':False,'state_path_accepted':False} for a in ARMS for c in CHANNELS for f in FAMILIES for k in KEYS for m in MODES},
 path_posthoc={a+'/'+c+'/'+f+'/'+k+'/'+m:{'status':'NOT_RUN_SETUP'} for a in ARMS for c in CHANNELS for f in FAMILIES for k in KEYS for m in MODES},
 edge_posthoc={a+'/'+c+'/'+f+'/'+k:{'status':'NOT_RUN_SETUP'} for a in ARMS for c in CHANNELS for f in FAMILIES for k in KEYS},
 payload_reads={a+'/'+c+'/'+k:{'status':'NOT_RUN_SETUP'} for a in ARMS for c in CHANNELS for k in KEYS},
 payload_evaluations={a+'/'+c+'/'+k+'/'+t:{'status':'NOT_RUN_SETUP'} for a in ARMS for c in CHANNELS for k in KEYS for t in ('REGISTERED','WRONG_MESSAGE')},
 quality={c+'/'+a+'_vs_'+b:{'status':'NOT_RUN_SETUP'} for c in CHANNELS for i,a in enumerate(ARMS) for b in ARMS[:i]})

setup['group_observations']={a+'/'+c+'/'+k:{'status':'NOT_RUN_SETUP'} for a in ARMS for c in CHANNELS for k in KEYS}
setup['state_budget_schedule']={'status':'NOT_RUN_SETUP','source_arm':'STATE_OLD_MULTI','target_arm':'STATE_REPLICA_MULTI','role':'writer only; never receiver input','steps':{str(i):{'status':'NOT_RUN_SETUP','target_l2':None} for i in range(25,50)}}
setup['terminal_diagnostic_fixed']=TERMINAL_FIXED
setup['terminal_diagnostics']=dict(
 inputs={a:{'status':'NOT_RUN_SETUP'} for a in ARMS},
 group_observations={a+'/'+k:{'status':'NOT_RUN_SETUP'} for a in ARMS for k in KEYS},
 projections={a+'/'+f+'/'+k:{'status':'NOT_RUN_SETUP'} for a in ARMS for f in FAMILIES for k in KEYS},
 mode_reads={a+'/'+f+'/'+k+'/'+m:{'status':'NOT_RUN_SETUP','accepted_payload':False,'state_path_accepted':False} for a in ARMS for f in FAMILIES for k in KEYS for m in MODES},
 path_posthoc={a+'/'+f+'/'+k+'/'+m:{'status':'NOT_RUN_SETUP'} for a in ARMS for f in FAMILIES for k in KEYS for m in MODES},
 edge_posthoc={a+'/'+f+'/'+k:{'status':'NOT_RUN_SETUP'} for a in ARMS for f in FAMILIES for k in KEYS},
 counts={},calls={},role='writer diagnostic only; never primary blind input')
def write_json(path,value):
    tmp=path.with_suffix(path.suffix+'.tmp');tmp.write_text(json.dumps(value,indent=2)+'\n');os.replace(tmp,path)
def failed(stage,exc):
    setup.update(status='SETUP_FAILED',stage=stage,error=f'{type(exc).__name__}: {exc}')
    write_json(OUTPUT/'setup_receipt.json',setup)
    write_json(OUTPUT/'setup_failure.json',dict(stage=stage,error=str(exc),traceback=traceback.format_exc()))
def logged(command,stage,cwd=None,check=True):
    try:
        with (OUTPUT/'execution.log').open('a') as log:
            log.write('COMMAND '+repr(command)+'\n');log.flush()
            child=subprocess.Popen(command,cwd=cwd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
            for line in child.stdout:print(line,end='',flush=True);log.write(line);log.flush()
            code=child.wait();log.write('EXIT '+str(code)+'\n')
        if check and code:raise subprocess.CalledProcessError(code,command)
        return code
    except Exception as exc:failed(stage,exc);raise
write_json(OUTPUT/'setup_receipt.json',setup)
print('Fixed output:',OUTPUT)
if SOURCE_SHA is None:
    error=RuntimeError('Unpublished draft: bind published source SHA before Run all');failed('SOURCE_BINDING',error);raise error


In [ ]:
try:
    logged(['git','clone','--filter=blob:none','https://github.com/RICHAAARC/SC-SSTW.git',str(REPO)],'SOURCE_CLONE')
    logged(['git','-C',str(REPO),'fetch','origin',SOURCE_SHA],'SOURCE_FETCH')
    logged(['git','-C',str(REPO),'checkout','--detach',SOURCE_SHA],'SOURCE_CHECKOUT')
    actual=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
    dirty=subprocess.check_output(['git','-C',str(REPO),'status','--porcelain'],text=True)
    if actual!=SOURCE_SHA or dirty:raise RuntimeError('source SHA/clean checkout mismatch')
    write_json(OUTPUT/'source_receipt.json',dict(source_sha=actual,clean=True))
    import shutil
    tools={name:shutil.which(name) for name in ('ffmpeg','ffprobe')}
    tool_codes={name:logged([path,'-version'],name.upper()+'_PROBE',check=False) if path else None for name,path in tools.items()}
    if any(code!=0 for code in tool_codes.values()):
        try:
            logged(['apt-get','update'],'MEDIA_TOOL_UPDATE',check=False)
            logged(['apt-get','install','-y','ffmpeg'],'MEDIA_TOOL_REPAIR',check=False)
        except Exception as repair_error:print('Media-tool repair error retained:',repair_error)
    setup['media_tool_probe_returncodes']=tool_codes
    torch_ok=logged([PYTHON,'-c','import torch; print(torch.__version__)'],'TORCH_PROBE',check=False)==0
    if not torch_ok:logged([PYTHON,'-m','pip','install','--upgrade','torch'],'TORCH_REPAIR',check=False)
    vae_probe='from diffusers import AutoencoderKLWan,WanPipeline; print(AutoencoderKLWan.__name__,WanPipeline.__name__)'
    vae_ok=logged([PYTHON,'-c',vae_probe],'VAE_IMPORT_PROBE',check=False)==0
    if not vae_ok:
        logged([PYTHON,'-m','pip','install','diffusers>=0.35.1,<0.40','transformers>=4.45,<5','accelerate>=0.34','safetensors>=0.4.5'],'VAE_DEPENDENCY_REPAIR',check=False)
    dependency_code=logged([PYTHON,'-m','pip','check'],'DEPENDENCY_REPORT',check=False)
    setup['dependency_check_returncode']=dependency_code
    if dependency_code:print('Metadata conflicts retained; actual VAE import/load determines availability.')
    (OUTPUT/'environment_freeze.txt').write_text(subprocess.check_output([PYTHON,'-m','pip','freeze'],text=True))
    environment_code='import json,torch; from experiments.wan_state_clock.video_local_fourier_rm_replica_v1_run import environment; r=environment(); r.update(cuda_available=torch.cuda.is_available()); print(json.dumps(r))'
    env=json.loads(subprocess.check_output([PYTHON,'-c',environment_code],cwd=REPO,text=True))
    write_json(OUTPUT/'environment_receipt.json',env)
    if not env['cuda_available']:print('CUDA unavailable: CPU path selected; much slower.')
    setup.update(status='SETUP_COMPLETE',python_executable=PYTHON,environment_strategy='current_python_separate_generation_media_workers')
    write_json(OUTPUT/'setup_receipt.json',setup)
except Exception as exc:failed('SOURCE_OR_ENVIRONMENT',exc);raise


In [ ]:
try:
    from google.colab import userdata
    token=userdata.get('HF_TOKEN')
    if token:os.environ['HF_TOKEN']=token
except Exception:
    pass
finally:
    token=None
command=[PYTHON,'-u','-m','experiments.wan_state_clock.video_local_fourier_rm_replica_v1_run','--output',str(RUN_OUTPUT)]
try:
    returncode=logged(command,'FIXED_RUN',cwd=REPO,check=False)
    RESULT_PATH=RUN_OUTPUT/'result.json'
    write_json(OUTPUT/'execution_receipt.json',dict(command=command,returncode=returncode,result_path=str(RESULT_PATH),result_exists=RESULT_PATH.is_file()))
    if not RESULT_PATH.is_file():raise RuntimeError('No runner result; setup denominator retained')
    setup.update(status='SUPERSEDED_BY_RESULT',result_path=str(RESULT_PATH));write_json(OUTPUT/'setup_receipt.json',setup)
except Exception as exc:failed('RUNNER',exc);raise


In [ ]:
result=json.loads(RESULT_PATH.read_text())
if result['source_sha']!=SOURCE_SHA or result['fixed_denominator']!=FIXED:raise RuntimeError('result identity mismatch')
print('Status:',result['status'],'Counts:',result['counts'])
print('Calls:',result['calls'],'Workers:',result['workers'])
print('Shared before25:',result.get('before_step25_identity'))
budget=result.get('state_budget_schedule',{})
print('State budget schedule:',budget)
for arm,row in result['generation'].items():
    for step in row.get('steps',[]):
        if arm.startswith('STATE_') and step.get('index') in (25,49):
            print('Applied state budget:',arm,step['index'],step.get('pilot'))
terminal=result.get('terminal_diagnostics',{})
print('Writer terminal diagnostic counts/calls:',terminal.get('counts'),terminal.get('calls'))
for name,row in terminal.get('path_posthoc',{}).items():
    print('Writer terminal only:',name,row['status'],'rank=',row.get('true_path_rank'),'Delta=',row.get('true_path_delta'),'matches=',row.get('canonical_path_matches'))
print('Group responses retained:',len(result.get('group_observations',{})),'primary and',len(terminal.get('group_observations',{})),'terminal records; correlated, not independent samples')

for arm,row in result['generation'].items():print('Writer:',arm,row['status'],(row.get('receipt') or {}).get('control_norm_statistics'),(row.get('receipt') or {}).get('writer_diagnostics'))
for family in FAMILIES:
    print('Independent reader family:',family)
    for name,row in result['path_posthoc'].items():
        if '/'+family+'/' in name:print(name,row['status'],'rank=',row.get('true_path_rank'),'Delta=',row.get('true_path_delta'),'matches=',row.get('canonical_path_matches'))
for name,row in result['payload_evaluations'].items():print('Repeated payload:',name,row['status'],'bit errors=',row.get('bit_errors'),'exact=',row.get('exact_bits'))
print('Saved-RGB quality diagnostics:',result['quality'])
from IPython.display import Video,display
for arm in ARMS:
    video=result['transport'].get(arm+'/MP4',{}).get('events',{}).get('mp4',{})
    path=video.get('path')
    if video.get('status')=='SAVED' and isinstance(path,str) and Path(path).is_file():
        try:print('Saved MP4 preview:',arm);display(Video(str(path),embed=True))
        except Exception as preview_error:print('Preview unavailable; result retained:',arm,preview_error)
print('Failures:',result['failures'])
print('Evidence ceiling:',result['evidence_ceiling'])
print('Retained result:',RESULT_PATH)
